In [ ]:
import random

from tqdm import tqdm
from glob import glob
from pathlib import Path

from rich.pretty import pprint
from rich.markdown import Markdown

from rage.retriever import Retriever
from rage.splitters import MarkdownSplitter
from rage.utils.embeddings import get_openai_embeddings
from rage.loaders import PDFMarkdownLoader, MarkdownLoader, DocxLoader

In [ ]:
loader_map = {
    ".pdf": PDFMarkdownLoader(),
    ".docx": DocxLoader(),
    ".epub": MarkdownLoader(),
    ".rtf": MarkdownLoader(),
    ".txt": MarkdownLoader(),
}

In [ ]:
splitter = MarkdownSplitter()
retriever = Retriever(dense_embeddings=get_openai_embeddings())

In [ ]:
file_paths = glob("/resources/documents/material-interactions/material/*")
file_paths = [
    {
        "path": fp,
        "extension": Path(fp).suffix,
    }
    for fp in file_paths
]

print(f"file_paths => {len(file_paths)}")
pprint(file_paths[:5])

In [ ]:
documents = []
for file_path in tqdm(file_paths):
    extension = file_path["extension"]  # type: ignore
    loader = loader_map[extension]

    path = file_path["path"]  # type: ignore
    documents.extend(await loader.load(source_path=path))

In [ ]:
text_chunks = splitter.split_documents(documents=documents)
print(f"text_chunks => {len(text_chunks)}")

In [ ]:
text_chunk = random.choice(text_chunks)
pprint(text_chunk.metadata)
Markdown(text_chunk.text)

In [ ]:
collection_name = "matter"
await retriever.create_collection(collection_name=collection_name)
await retriever.insert_text_chunks(
    collection_name=collection_name,
    text_chunks=text_chunks,
)